In [2]:
import numpy as np
import pandas as pd
import pickle
from itertools import combinations

In [3]:
df_name = 'dbpedia'
model_name = 'ctm'

topics_dict_path = f'saved/{df_name}_{model_name}/{df_name}_topics_dict.pkl'

with open(topics_dict_path, "rb") as f:
    topics_dict = pickle.load(f)

topics_dict

{0: [('sir', 0),
  ('daughter', 0),
  ('son', 0),
  ('lord', 0),
  ('wife', 0),
  ('baron', 0),
  ('henry', 0),
  ('count', 0),
  ('earl', 0),
  ('duke', 0)],
 1: [('train', 0),
  ('trains', 0),
  ('seoul', 0),
  ('gauge', 0),
  ('railways', 0),
  ('dong', 0),
  ('line', 0),
  ('korea', 0),
  ('canton', 0),
  ('sen', 0)],
 2: [('placing', 0),
  ('close', 0),
  ('contest', 0),
  ('preceding', 0),
  ('night', 0),
  ('entry', 0),
  ('performed', 0),
  ('voting', 0),
  ('succeeded', 0),
  ('received', 0)],
 3: [('mixed', 0),
  ('crowned', 0),
  ('ultimate', 0),
  ('fighting', 0),
  ('sumo', 0),
  ('beauty', 0),
  ('thai', 0),
  ('competitor', 0),
  ('fights', 0),
  ('martial', 0)],
 4: [('independent', 0),
  ('bands', 0),
  ('metal', 0),
  ('records', 0),
  ('rock', 0),
  ('releases', 0),
  ('artists', 0),
  ('jazz', 0),
  ('sound', 0),
  ('electronic', 0)],
 5: [('council', 0),
  ('riding', 0),
  ('conservative', 0),
  ('quebec', 0),
  ('canadian', 0),
  ('commons', 0),
  ('re', 0),
  ('c

In [4]:
def calculate_topic_diversity(topics_dict, top_k=10):
    print(f"Loading topic dictionary for {df_name}...")

    # 2. Extract the words
    all_topic_words = []
    
    for t_id, word_weight_tuples in topics_dict.items():
        # Ignore the noise topic (-1) and empty topics
        if t_id != -1 and len(word_weight_tuples) > 0:
            # Grab only the Top K words for this topic (usually 10)
            top_words = [word for word, weight in word_weight_tuples[:top_k]]
            all_topic_words.extend(top_words)

    # 3. Calculate Diversity
    total_words = len(all_topic_words)
    unique_words = len(set(all_topic_words))
    
    if total_words == 0:
        print("Error: No valid topics found.")
        return 0.0

    diversity_score = unique_words / total_words

    print(f"\n--- Diversity Results ---")
    print(f"Total Topics Evaluated: {len(topics_dict) - (1 if -1 in topics_dict else 0)}")
    print(f"Total Words (Top {top_k}): {total_words}")
    print(f"Unique Words: {unique_words}")
    print(f"Global Topic Diversity (TD): {diversity_score:.4f}")

    return diversity_score

In [5]:
diversity_score = calculate_topic_diversity(topics_dict, top_k=10)

diversity_score

Loading topic dictionary for dbpedia...

--- Diversity Results ---
Total Topics Evaluated: 500
Total Words (Top 10): 5000
Unique Words: 1840
Global Topic Diversity (TD): 0.3680


0.368

In [6]:
relevant_topics_path = f"saved/{df_name}_{model_name}/{df_name}_relevant_topics.pkl"


with open(relevant_topics_path, "rb") as f:
    relevant_topics_for_all = pickle.load(f)

relevant_topics_for_all[:5]

[[(np.int64(376), np.float32(0.11505146)),
  (np.int64(40), np.float32(0.09470468)),
  (np.int64(446), np.float32(0.088142276))],
 [(np.int64(441), np.float32(0.078045)),
  (np.int64(126), np.float32(0.07122444)),
  (np.int64(105), np.float32(0.06763124))],
 [(np.int64(340), np.float32(0.05726418)),
  (np.int64(499), np.float32(0.041172963)),
  (np.int64(333), np.float32(0.03644754))],
 [(np.int64(468), np.float32(0.35153264)),
  (np.int64(314), np.float32(0.18016408)),
  (np.int64(294), np.float32(0.12644267))],
 [(np.int64(167), np.float32(0.074640356)),
  (np.int64(437), np.float32(0.048589416)),
  (np.int64(90), np.float32(0.029654821))]]

In [7]:
final_topics_path = f"saved/{df_name}_{model_name}/{df_name}_final_topics.pkl"


with open(final_topics_path, "rb") as f:
    final_topics_for_all = pickle.load(f)

final_topics_for_all[:5]

[[(np.int64(376), np.float32(0.11505146)),
  (np.int64(446), np.float32(0.088142276))],
 [(np.int64(441), np.float32(0.078045))],
 [(np.int64(340), np.float32(0.05726418)),
  (np.int64(499), np.float32(0.041172963)),
  (np.int64(333), np.float32(0.03644754))],
 [(np.int64(468), np.float32(0.35153264))],
 [(np.int64(167), np.float32(0.074640356))]]

In [8]:
def calculate_document_diversity(topics_dict, relevant_topics_for_all, top_k=10):

    document_diversity_scores = []
    skipped_docs = 0

    print("Calculating Pairwise Document-Level Diversity...")

    # 3. Iterate through every document's predicted topics
    for doc_topics in relevant_topics_for_all:
        
        # Extract just the topic IDs, ignoring the noise topic (-1)
        valid_topic_ids = [t_id for t_id, prob in doc_topics if t_id != -1]

        # If a document has 0 or 1 topic, we can't calculate pairwise diversity.
        if len(valid_topic_ids) < 2:
            skipped_docs += 1
            continue

        # Extract the top K words for each valid topic as a Python Set
        topic_word_sets = []
        for t_id in valid_topic_ids:
            word_weights = topics_dict.get(t_id, [])
            words_only = set([word for word, weight in word_weights[:top_k]])
            if len(words_only) > 0:
                topic_word_sets.append(words_only)

        # Safety check in case topics were empty
        if len(topic_word_sets) < 2:
            skipped_docs += 1
            continue

        # 4. Calculate Pairwise Jaccard Similarity for this document
        doc_jaccard_scores = []
        # combinations() generates every unique pair (e.g., A-B, A-C, B-C)
        for set_a, set_b in combinations(topic_word_sets, 2):
            intersection = len(set_a.intersection(set_b))
            union = len(set_a.union(set_b))
            
            sim = intersection / union if union > 0 else 0.0
            doc_jaccard_scores.append(sim)

        # 5. Document Diversity = 1.0 - Average Overlap
        avg_doc_overlap = np.mean(doc_jaccard_scores)
        doc_diversity = 1.0 - avg_doc_overlap
        document_diversity_scores.append(doc_diversity)

    # 6. Final Global Average
    if len(document_diversity_scores) == 0:
        print("Error: No documents had 2 or more valid topics.")
        return 0.0

    final_average_diversity = np.mean(document_diversity_scores)

    print(f"\n--- Document-Level Diversity Results ---")
    print(f"Total Documents Evaluated: {len(relevant_topics_for_all)}")
    print(f"Multi-Topic Docs Scored: {len(document_diversity_scores)}")
    print(f"Single-Topic Docs Skipped: {skipped_docs}")
    print(f"Average Document-Level Diversity: {final_average_diversity:.4f}")

    return final_average_diversity

In [9]:
diversity_pre_selection = calculate_document_diversity(topics_dict, relevant_topics_for_all, top_k=10)

diversity_pre_selection

Calculating Pairwise Document-Level Diversity...

--- Document-Level Diversity Results ---
Total Documents Evaluated: 337739
Multi-Topic Docs Scored: 336826
Single-Topic Docs Skipped: 913
Average Document-Level Diversity: 0.6790


np.float64(0.6789753794137955)

In [10]:
diversity_post_selection = calculate_document_diversity(topics_dict, final_topics_for_all, top_k=10)

diversity_post_selection

Calculating Pairwise Document-Level Diversity...

--- Document-Level Diversity Results ---
Total Documents Evaluated: 337739
Multi-Topic Docs Scored: 223885
Single-Topic Docs Skipped: 113854
Average Document-Level Diversity: 0.9069


np.float64(0.906934588218899)